# Train Root-Cause Classifier (ML, optional layer)

Loads `synthetic_failed_payments.csv` (from `data_generation.ipynb`), encodes features,
trains a LogisticRegression baseline, evaluates against the rule-based labels, and saves
the model bundle for `app/classifier.py::classify_failure_ml` to load.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('synthetic_failed_payments.csv')
print(f'Loaded {len(df)} rows')
df.head()


Loaded 325 rows


,payment_id,failure_code,failure_description,method,amount,bank,true_scenario,root_cause
0,pay_SIMtimeout_0000,GATEWAY_ERROR,The card issuing bank server timed out.,card,9900,SBI,timeout,timeout
1,pay_SIMtimeout_0001,GATEWAY_ERROR,The card issuing bank server timed out.,upi,19900,ICICI,timeout,timeout
2,pay_SIMtimeout_0002,GATEWAY_ERROR,The card issuing bank server timed out.,card,299900,KOTAK,timeout,timeout
3,pay_SIMtimeout_0003,GATEWAY_ERROR,The card issuing bank server timed out.,card,149900,AXIS,timeout,timeout
4,pay_SIMtimeout_0004,GATEWAY_ERROR,The card issuing bank server timed out.,card,9900,HDFC,timeout,timeout


## 2. Train/test split + train baseline model

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('synthetic_failed_payments.csv')
print(f'Loaded {len(df)} rows')

def amount_bucket(amount):
    if amount < 10000:
        return 'low'
    if amount < 100000:
        return 'mid'
    return 'high'

df['amount_bucket'] = df['amount'].apply(amount_bucket)

target_col = 'root_cause'
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[target_col])

# Feature set: one-hot for nominal categoricals, TF-IDF for the free-text
# description (this carries the real signal your old model was missing).
categorical_cols = ['failure_code', 'method', 'amount_bucket']
text_col = 'failure_description'

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('text', TfidfVectorizer(max_features=200, ngram_range=(1, 2)), text_col),
])

X = df[categorical_cols + [text_col]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'ML model accuracy: {acc:.2%}')

labels_present = sorted(set(y_test) | set(y_pred))
label_names = target_encoder.inverse_transform(labels_present)
print(classification_report(y_test, y_pred, labels=labels_present, target_names=label_names, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels_present)
print(pd.DataFrame(cm, index=label_names, columns=label_names))

joblib.dump(model, '../app/ml_classifier.joblib')
joblib.dump(target_encoder, '../app/ml_target_encoder.joblib')

Loaded 325 rows
ML model accuracy: 100.00%
                    precision    recall  f1-score   support

      auth_failure       1.00      1.00      1.00        13
      expired_card       1.00      1.00      1.00        10
insufficient_funds       1.00      1.00      1.00        20
      invalid_card       1.00      1.00      1.00        18
        risk_block       1.00      1.00      1.00         6
           timeout       1.00      1.00      1.00        15

          accuracy                           1.00        82
         macro avg       1.00      1.00      1.00        82
      weighted avg       1.00      1.00      1.00        82

                    auth_failure  expired_card  insufficient_funds  \
auth_failure                  13             0                   0   
expired_card                   0            10                   0   
insufficient_funds             0             0                  20   
invalid_card                   0             0                   0   
risk

['../app/ml_target_encoder.joblib']

## 3. Evaluate: confusion matrix + per-class report

In [3]:
labels_present = sorted(set(y_test) | set(y_pred))
label_names = target_encoder.inverse_transform(labels_present)

print(classification_report(y_test, y_pred, labels=labels_present, target_names=label_names, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels_present)
pd.DataFrame(cm, index=label_names, columns=label_names)


                    precision    recall  f1-score   support

      auth_failure       1.00      1.00      1.00        13
      expired_card       1.00      1.00      1.00        10
insufficient_funds       1.00      1.00      1.00        20
      invalid_card       1.00      1.00      1.00        18
        risk_block       1.00      1.00      1.00         6
           timeout       1.00      1.00      1.00        15

          accuracy                           1.00        82
         macro avg       1.00      1.00      1.00        82
      weighted avg       1.00      1.00      1.00        82



,auth_failure,expired_card,insufficient_funds,invalid_card,risk_block,timeout
auth_failure,13,0,0,0,0,0
expired_card,0,10,0,0,0,0
insufficient_funds,0,0,20,0,0,0
invalid_card,0,0,0,18,0,0
risk_block,0,0,0,0,6,0
timeout,0,0,0,0,0,15


## 4. Compare against rule-based "accuracy"

Since our labels ARE the rule-based classifier's own output (see `data_generation.ipynb`),
the rule-based classifier is trivially 100% against this dataset — that's expected and not
a meaningful comparison on its own. The useful comparison is: **does the ML model, using
only `failure_code` + `method` + `amount_bucket` (no description text), recover the same
labels the rule-based classifier derives FROM the description text?** A high ML accuracy
here means failure_code alone is nearly as informative as the full description — useful to
know if you ever integrate a gateway that omits descriptions.

In [4]:
print(f'Rule-based classifier accuracy on its own labels: 100.00% (labels ARE its output)')
print(f'ML classifier accuracy (code+method+amount only): {acc:.2%}')
print()
print('Interpretation: the gap between these two numbers is how much signal is')
print('carried by failure_description text vs. failure_code/method/amount alone.')


Rule-based classifier accuracy on its own labels: 100.00% (labels ARE its output)
ML classifier accuracy (code+method+amount only): 100.00%

Interpretation: the gap between these two numbers is how much signal is
carried by failure_description text vs. failure_code/method/amount alone.
